# ==============================================================================
# 📰 ENTERPRISE TOPIC CLASSIFICATION: BBC NEWS VIA MULTINOMIAL NAIVE BAYES
# ==============================================================================
### Mathematical Formulation:
Multinomial Naive Bayes models discrete word count or term-frequency feature vectors:
$$P(X \mid C_k) = \frac{(\sum_i x_i)!}{\prod_i x_i!} \prod_{i=1}^d p_{ki}^{x_i}$$
With Laplace (Add-1) Smoothing to prevent the Zero-Frequency Trap:
$$p_{ki} = \frac{N_{ki} + \alpha}{N_k + \alpha \cdot |V|}$$
where $N_{ki}$ is the count of term $i$ in category $k$, $N_k$ is the total terms in category $k$, $|V|$ is the vocabulary size, and $\alpha$ is the smoothing hyperparameter.

Crucial Rule: **NEVER USE `StandardScaler()`** on text matrices! StandardScaler introduces negative values which invalidate multinomial count assumptions. Instead, use `TfidfVectorizer(sublinear_tf=True)` or `CountVectorizer()`.


In [1]:
# STEP 1: IMPORTS & ENVIRONMENT SETUP
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("✅ STEP 1: Text NLP & MultinomialNB Environment ready.")


✅ STEP 1: Text NLP & MultinomialNB Environment ready.


## 📥 STEP 2 & 3: INGESTION & COLUMN SANITIZATION
Loading the Kaggle BBC News Archive (2,225 curated articles across 5 news categories).


In [2]:
# STEP 2 & 3: LOAD & SANITIZE
data_path = 'data/bbc_news/bbc-news-data.csv'
df = pd.read_csv(data_path, sep='\t')

df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
print(f"📊 Dataset Shape: {df.shape[0]:,} articles, {df.shape[1]} columns")
print(f"📋 Categories: {df['category'].value_counts().to_dict()}")
df[['category', 'title', 'content']].head(3)


📊 Dataset Shape: 2,225 articles, 4 columns
📋 Categories: {'sport': 511, 'business': 510, 'politics': 417, 'tech': 401, 'entertainment': 386}


,category,title,content
0,business,Ad sales boost Time Warner profit,Quarterly profits at US media giant TimeWarne...
1,business,Dollar gains on Greenspan speech,The dollar has hit its highest level against ...
2,business,Yukos unit buyer faces loan claim,The owners of embattled Russian oil giant Yuk...


## 🧹 STEP 4 & 5: SEGREGATION & HYGIENE
Combining headline and body text into a unified document feature, and handling missing/duplicate content.


In [3]:
# STEP 4 & 5: TEXT PREPARATION & HYGIENE
df['full_text'] = df['title'].fillna('') + ' ' + df['content'].fillna('')

X = df['full_text']
y = df['category']

print(f"Missing documents: {X.isnull().sum()}")
print(f"Class distribution:\n{y.value_counts(normalize=True).round(4) * 100}")


Missing documents: 0
Class distribution:
category
sport            22.97
business         22.92
politics         18.74
tech             18.02
entertainment    17.35
Name: proportion, dtype: float64


## 🔒 STEP 6: STRATIFIED SPLIT
Stratified 80/20 train/test split preserving topic representation.


In [4]:
# STEP 6: TRAIN-TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Training Documents : {len(X_train):,}")
print(f"Testing Documents  : {len(X_test):,}")


Training Documents : 1,780
Testing Documents  : 445


## ⚙️ STEP 7: TF-IDF VECTORIZATION PIPELINE
Using `TfidfVectorizer` with `sublinear_tf=True` (logarithmic term frequency scaling: $1 + \log(tf)$)
and unigram + bigram n-grams.


In [5]:
# STEP 7: NLP PIPELINE
tfidf = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1, 2),
    max_features=10000,
    sublinear_tf=True
)
print("✅ STEP 7: Sublinear TF-IDF Vectorizer initialized.")


✅ STEP 7: Sublinear TF-IDF Vectorizer initialized.


## ⚠️ STEP 8: THE ZERO-FREQUENCY TRAP & LAPLACE SMOOTHING
Comparing $\alpha=0$ (unsmoothed, fails on unseen test words) vs. $\alpha=1.0$ (Laplace smoothed).


In [6]:
# STEP 8: ZERO-FREQUENCY COMPARISON
pipe_alpha_zero = Pipeline([('tfidf', tfidf), ('mnb', MultinomialNB(alpha=1e-9))])
pipe_alpha_zero.fit(X_train, y_train)

pipe_laplace = Pipeline([('tfidf', tfidf), ('mnb', MultinomialNB(alpha=1.0))])
pipe_laplace.fit(X_train, y_train)

acc_zero = accuracy_score(y_test, pipe_alpha_zero.predict(X_test))
acc_laplace = accuracy_score(y_test, pipe_laplace.predict(X_test))

print(f"Unsmoothed (alpha=1e-9) Test Accuracy : {acc_zero*100:.2f}%")
print(f"Laplace Smoothed (alpha=1.0) Test Accuracy: {acc_laplace*100:.2f}%")


Unsmoothed (alpha=1e-9) Test Accuracy : 97.75%
Laplace Smoothed (alpha=1.0) Test Accuracy: 99.10%


## 🎯 STEP 9: HYPERPARAMETER TUNING (`alpha`)
Fine-tuning the smoothing parameter $\alpha$ across 15 values using Stratified 5-Fold Cross-Validation.


In [7]:
# STEP 9: GRID SEARCH FOR ALPHA
param_grid = {
    'mnb__alpha': np.logspace(-3, 1, 15)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(
    Pipeline([('tfidf', tfidf), ('mnb', MultinomialNB())]),
    param_grid=param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1
)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print(f"🏆 Best Alpha Parameter: {grid.best_params_}")
print(f"🎯 Best 5-Fold CV Accuracy: {grid.best_score_*100:.2f}%")


🏆 Best Alpha Parameter: {'mnb__alpha': np.float64(0.013894954943731374)}
🎯 Best 5-Fold CV Accuracy: 97.75%


## ⚔️ STEP 10: ALGORITHM DUEL (MultinomialNB vs Logistic Regression)
Evaluating MultinomialNB against L2-regularized Logistic Regression.


In [8]:
# STEP 10: MODEL BENCHMARK
pipe_lr = Pipeline([
    ('tfidf', tfidf),
    ('lr', LogisticRegression(max_iter=1000, random_state=42))
])
pipe_lr.fit(X_train, y_train)

y_pred_nb = best_model.predict(X_test)
y_pred_lr = pipe_lr.predict(X_test)

print("="*65)
print(f"MultinomialNB (Tuned)  -> Test Accuracy: {accuracy_score(y_test, y_pred_nb)*100:.2f}% | Macro F1: {f1_score(y_test, y_pred_nb, average='macro')*100:.2f}%")
print(f"Logistic Regression    -> Test Accuracy: {accuracy_score(y_test, y_pred_lr)*100:.2f}% | Macro F1: {f1_score(y_test, y_pred_lr, average='macro')*100:.2f}%")
print("="*65)


MultinomialNB (Tuned)  -> Test Accuracy: 98.20% | Macro F1: 98.15%
Logistic Regression    -> Test Accuracy: 98.88% | Macro F1: 98.87%


## 📊 STEP 11: MULTICLASS DIAGNOSTICS & EVALUATION
Comprehensive classification metrics across all 5 news categories.


In [9]:
# STEP 11: EVALUATION REPORT
print("📋 BBC NEWS 5-TOPIC CLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred_nb))

cm = confusion_matrix(y_test, y_pred_nb)
print(f"Confusion Matrix:\n{cm}")


📋 BBC NEWS 5-TOPIC CLASSIFICATION REPORT:
               precision    recall  f1-score   support

     business       1.00      0.95      0.97       102
entertainment       0.97      0.99      0.98        77
     politics       0.96      0.98      0.97        84
        sport       1.00      1.00      1.00       102
         tech       0.96      1.00      0.98        80

     accuracy                           0.98       445
    macro avg       0.98      0.98      0.98       445
 weighted avg       0.98      0.98      0.98       445

Confusion Matrix:
[[ 97   1   2   0   2]
 [  0  76   1   0   0]
 [  0   1  82   0   1]
 [  0   0   0 102   0]
 [  0   0   0   0  80]]


## 💾 STEP 12 & 13: PRODUCTION SERIALIZATION & NEWS DESK SMOKE TEST
Deploying pipeline and testing article triage latency.


In [10]:
# STEP 12 & 13: SERIALIZATION & SMOKE TEST
os.makedirs('production_models', exist_ok=True)
model_path = 'production_models/bbc_news_multinomial_nb_pipeline.joblib'
joblib.dump(best_model, model_path)
print(f"💾 Production pipeline saved to: {model_path} ({os.path.getsize(model_path):,} bytes)")

loaded_pipe = joblib.load(model_path)
sample_article = [
    "Government announces new fiscal spending plan for renewable energy subsidies and corporate tax cuts."
]

t0 = time.perf_counter()
pred_cat = loaded_pipe.predict(sample_article)[0]
probs = loaded_pipe.predict_proba(sample_article)[0]
latency_ms = (time.perf_counter() - t0) * 1000

print(f"\n📰 LIVE ARTICLE TRIAGE SMOKE TEST:")
print(f"   Classified Topic : 🏷️ {pred_cat.upper()}")
print(f"   Confidence       : {probs.max()*100:.2f}%")
print(f"   Inference Latency: {latency_ms:.3f} ms (SLA < 2.0ms: {'PASS' if latency_ms < 2.0 else 'CHECK'})")


💾 Production pipeline saved to: production_models/bbc_news_multinomial_nb_pipeline.joblib (1,182,569 bytes)



📰 LIVE ARTICLE TRIAGE SMOKE TEST:
   Classified Topic : 🏷️ BUSINESS
   Confidence       : 92.50%
   Inference Latency: 2.080 ms (SLA < 2.0ms: CHECK)
